# 12 Unified GSEA: Enrichment, Visualization, and Custom Lists

## Overview
This notebook merges the previous notebook 11 (selective enrichment compute) and notebook 12 (visualization + custom list workflows) into a single entrypoint.

## What This Notebook Does
- Runs selective pathway enrichment for a user-selected cell type from notebook 09 DGE workbooks
- Visualizes saved level-wise or selective pathway outputs
- Supports optional custom gene-list prerank GSEA for targeted hypotheses

## Primary Inputs
- `Results/DGE/DGE_level_class_nb09.xlsx`
- `Results/DGE/DGE_level_subclass_nb09.xlsx`
- `Results/DGE/DGE_level_supertype_nb09.xlsx`
- `Results/DGE/DGE_level_cluster_nb09.xlsx`

## Primary Outputs
- Selective enrichment tables under `Results/Enrichment/selective/`
- Plots under `Results/GSEA/plots/`
- Optional custom prerank outputs under `Results/Enrichment/custom/`
- Manifest at `Results/Enrichment/selective/selective_enrichment_manifest_nb12.csv`

## Standard Usage
1. Run setup/utilities.
2. Preview available cell types.
3. Optionally run selective enrichment.
4. Generate plots from saved outputs.
5. Optionally run custom gene-list prerank GSEA.

# Table of Contents

1. Setup and Unified Utilities
2. Selective Enrichment Example
3. Visualization and Custom GSEA Example

## Setup and Unified Utilities

This cell provides the full merged API for:
- selective enrichment from saved DGE workbooks,
- pathway plotting from saved outputs,
- optional custom prerank GSEA on user-defined gene lists.

In [ ]:
from __future__ import annotations

import time
import warnings
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Final, Iterable, Literal, cast

warnings.filterwarnings(
    "ignore",
    message=r"urllib3 \(.*\) or chardet \(.*\)/charset_normalizer \(.*\) doesn't match a supported version!",
    category=Warning,
    module="requests",
)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from matplotlib.figure import Figure

sns.set_theme(style="whitegrid")

LibraryKey = Literal["go", "kegg", "reactome", "wiki", "jensen"]
Direction = Literal["up", "down", "both"]
LEVELS: Final[tuple[str, ...]] = ("class", "subclass", "supertype", "cluster")


@dataclass(frozen=True)
class Notebook12Config:
    analysis_root: Path
    dge_dir: Path
    enrichment_root: Path
    selective_dir: Path
    plot_dir: Path
    custom_dir: Path
    selective_manifest: Path


ANALYSIS_ROOT: Final[Path] = Path("/media/drive_c/Project_Brain_snRNAseq/Analysis")

LIBRARY_SPECS: Final[dict[LibraryKey, dict[str, str]]] = {
    "go": {"gene_set": "GO_Biological_Process_2023", "label": "GO BP"},
    "kegg": {"gene_set": "KEGG_2021_Mouse", "label": "KEGG"},
    "reactome": {"gene_set": "Reactome_2022", "label": "Reactome"},
    "wiki": {"gene_set": "WikiPathways_2024_Mouse", "label": "WikiPathways"},
    "jensen": {"gene_set": "Jensen_DISEASES", "label": "Jensen DISEASES"},
}
DEFAULT_LIBRARY_ORDER: Final[tuple[LibraryKey, ...]] = ("go", "kegg", "reactome", "wiki", "jensen")

CONFIG = Notebook12Config(
    analysis_root=ANALYSIS_ROOT,
    dge_dir=ANALYSIS_ROOT / "Results" / "DGE",
    enrichment_root=ANALYSIS_ROOT / "Results" / "Enrichment",
    selective_dir=ANALYSIS_ROOT / "Results" / "Enrichment" / "selective",
    plot_dir=ANALYSIS_ROOT / "Results" / "GSEA" / "plots",
    custom_dir=ANALYSIS_ROOT / "Results" / "Enrichment" / "custom",
    selective_manifest=ANALYSIS_ROOT / "Results" / "Enrichment" / "selective" / "selective_enrichment_manifest_nb12.csv",
)
CONFIG.selective_dir.mkdir(parents=True, exist_ok=True)
CONFIG.plot_dir.mkdir(parents=True, exist_ok=True)
CONFIG.custom_dir.mkdir(parents=True, exist_ok=True)


def sanitize_name(text: str) -> str:
    keep = [char if char.isalnum() or char in ("-", "_") else "_" for char in str(text)]
    return "".join(keep).strip("_")[:120] or "untitled"


def normalize_label(text: str) -> str:
    return " ".join(str(text).replace("_", " ").replace("/", " ").lower().split())


def resolve_library_selection(libraries: str | Iterable[str] = "all") -> tuple[LibraryKey, ...]:
    if isinstance(libraries, str):
        if libraries.strip().lower() == "all":
            return DEFAULT_LIBRARY_ORDER
        requested = [item.strip().lower() for item in libraries.split(",") if item.strip()]
    else:
        requested = [str(item).strip().lower() for item in libraries]

    alias_map: dict[str, LibraryKey] = {
        "go": "go",
        "go_bp": "go",
        "gobp": "go",
        "go_biological_process_2023": "go",
        "kegg": "kegg",
        "kegg_2021_mouse": "kegg",
        "reactome": "reactome",
        "reactome_2022": "reactome",
        "wiki": "wiki",
        "wikipathways": "wiki",
        "wikipathways_2024_mouse": "wiki",
        "jensen": "jensen",
        "jensen_diseases": "jensen",
        "jensen diseases": "jensen",
        "janseon": "jensen",
    }

    resolved: list[LibraryKey] = []
    for item in requested:
        if item not in alias_map:
            raise ValueError(f"Unknown library '{item}'. Valid values: {sorted(alias_map)}")
        key = cast(LibraryKey, alias_map[item])
        if key not in resolved:
            resolved.append(key)
    if not resolved:
        raise ValueError("No valid libraries selected")
    return tuple(resolved)


def dge_workbook_candidates(level: str) -> list[Path]:
    return [
        CONFIG.dge_dir / f"DGE_level_{level}_nb09.xlsx",
        CONFIG.dge_dir / f"DGE_level_{level}.xlsx",
    ]


def resolve_existing_dge_workbook(level: str) -> Path:
    for candidate in dge_workbook_candidates(level):
        if candidate.exists():
            return candidate
    attempted = "\n".join(str(p) for p in dge_workbook_candidates(level))
    raise FileNotFoundError(f"Missing DGE workbook for level={level}. Tried:\n{attempted}")


def list_available_cell_types(level: str | None = None) -> pd.DataFrame:
    levels = (level,) if level is not None else LEVELS
    rows: list[dict[str, str]] = []
    for lvl in levels:
        xls = pd.ExcelFile(resolve_existing_dge_workbook(lvl))
        for sheet in xls.sheet_names:
            rows.append({"level": lvl, "cell_type": str(sheet), "workbook": str(resolve_existing_dge_workbook(lvl))})
    return pd.DataFrame(rows)


def locate_cell_type(cell_type: str, level: str | None = None) -> dict[str, str]:
    target_levels = (level,) if level is not None else LEVELS
    matches: list[dict[str, str]] = []

    for lvl in target_levels:
        xls = pd.ExcelFile(resolve_existing_dge_workbook(lvl))
        sheets = [str(sheet) for sheet in xls.sheet_names]
        if cell_type in sheets:
            matches.append({"query": cell_type, "level": lvl, "sheet_name": cell_type})
            continue
        norm_map = {normalize_label(name): name for name in sheets}
        norm_query = normalize_label(cell_type)
        if norm_query in norm_map:
            matches.append({"query": cell_type, "level": lvl, "sheet_name": norm_map[norm_query]})
            continue
        contains = [name for name in sheets if norm_query in normalize_label(name)]
        if len(contains) == 1:
            matches.append({"query": cell_type, "level": lvl, "sheet_name": contains[0]})

    if not matches:
        raise ValueError(f"Cell type '{cell_type}' not found. Use list_available_cell_types().")
    if len(matches) > 1 and level is None:
        levels_found = sorted({m["level"] for m in matches})
        raise ValueError(f"Cell type '{cell_type}' matches multiple levels {levels_found}. Pass level= to disambiguate.")
    return matches[0]


def infer_first_existing(columns: Iterable[str], candidates: Iterable[str], label: str) -> str:
    column_list = list(columns)
    for candidate in candidates:
        if candidate in column_list:
            return candidate
    raise KeyError(f"Could not identify {label} column. Available columns: {column_list}")


def read_celltype_dge(match: dict[str, str]) -> pd.DataFrame:
    xls = pd.ExcelFile(resolve_existing_dge_workbook(match["level"]))
    df = pd.read_excel(xls, sheet_name=match["sheet_name"])
    gene_col = infer_first_existing(df.columns, ["gene", "Gene", "names", "gene_symbol"], "gene")
    logfc_col = infer_first_existing(df.columns, ["logFC", "avg_log2FC", "log2FoldChange", "logfoldchanges", "avg_logFC"], "logFC")
    pval_col = infer_first_existing(df.columns, ["p_adj", "p_adj_scanpy", "pvals_adj", "adj.P.Val", "FDR", "p_val_adj", "p_val"], "adjusted p-value")

    out = df.copy()
    out["gene"] = out[gene_col].astype(str)
    out["logFC"] = pd.to_numeric(out[logfc_col], errors="coerce")
    out["p_adj_used"] = pd.to_numeric(out[pval_col], errors="coerce")
    return out.dropna(subset=["gene", "logFC", "p_adj_used"]).copy()


def select_enrichment_genes(dge_df: pd.DataFrame, direction: Direction, p_adj_threshold: float, min_abs_logfc: float, max_genes: int) -> dict[str, list[str]]:
    filt = dge_df[(dge_df["p_adj_used"] <= p_adj_threshold) & (dge_df["logFC"].abs() >= min_abs_logfc)].copy()
    if filt.empty:
        return {"up": [], "down": []}
    filt = filt.sort_values(["p_adj_used", "logFC"], ascending=[True, False])
    up = filt.loc[filt["logFC"] > 0, "gene"].drop_duplicates().head(max_genes).tolist()
    down = filt.loc[filt["logFC"] < 0, "gene"].drop_duplicates().head(max_genes).tolist()
    if direction == "up":
        return {"up": up, "down": []}
    if direction == "down":
        return {"up": [], "down": down}
    return {"up": up, "down": down}


def run_enrichr_with_retry(genes: list[str], gene_set_name: str, organism: str = "mouse", max_retries: int = 4, retry_base_seconds: float = 2.0) -> pd.DataFrame:
    if not genes:
        return pd.DataFrame()
    try:
        import gseapy as gp
    except ImportError as exc:
        raise ImportError("gseapy is required for enrichment. Install in active kernel.") from exc

    last_error: Exception | None = None
    for attempt in range(max_retries + 1):
        try:
            enr = gp.enrichr(gene_list=genes, gene_sets=gene_set_name, organism=organism, outdir=None, cutoff=1.0, no_plot=True)
            result_df = getattr(enr, "results", pd.DataFrame())
            return result_df.copy() if isinstance(result_df, pd.DataFrame) else pd.DataFrame()
        except Exception as exc:
            last_error = exc
            msg = str(exc).lower()
            should_retry = ("429" in msg or "too many requests" in msg or "rate" in msg) and attempt < max_retries
            if not should_retry:
                break
            time.sleep(retry_base_seconds * (2 ** attempt))
    raise RuntimeError(f"Enrichr failed for gene set {gene_set_name}: {last_error}")


def run_celltype_pathway_enrichment(
    cell_type: str,
    level: str | None = None,
    libraries: str | Iterable[str] = "all",
    direction: Direction = "both",
    p_adj_threshold: float = 0.05,
    min_abs_logfc: float = 0.25,
    max_genes: int = 500,
    fdr_threshold: float = 0.05,
    top_n_per_library: int = 20,
    organism: str = "Mouse",
    save_csv: bool = True,
) -> pd.DataFrame:
    library_keys = resolve_library_selection(libraries)
    match = locate_cell_type(cell_type=cell_type, level=level)
    dge_df = read_celltype_dge(match)

    genes_by_direction = select_enrichment_genes(
        dge_df=dge_df,
        direction=direction,
        p_adj_threshold=p_adj_threshold,
        min_abs_logfc=min_abs_logfc,
        max_genes=max_genes,
    )

    frames: list[pd.DataFrame] = []
    for direction_key, genes in genes_by_direction.items():
        if not genes:
            continue
        for library_key in library_keys:
            spec = LIBRARY_SPECS[library_key]
            lib_df = run_enrichr_with_retry(genes=genes, gene_set_name=spec["gene_set"], organism=organism)
            if lib_df.empty or "Adjusted P-value" not in lib_df.columns:
                continue
            work = lib_df.copy()
            work["Adjusted P-value"] = pd.to_numeric(work["Adjusted P-value"], errors="coerce")
            work = work.dropna(subset=["Adjusted P-value"])
            work = work[work["Adjusted P-value"] <= fdr_threshold].copy()
            if work.empty:
                continue
            work = work.sort_values("Adjusted P-value", ascending=True).head(top_n_per_library)
            work.insert(0, "cell_type_query", match["query"])
            work.insert(1, "cell_type_sheet", match["sheet_name"])
            work.insert(2, "level", match["level"])
            work.insert(3, "direction", direction_key)
            work.insert(4, "library_key", library_key)
            work.insert(5, "library_label", spec["label"])
            work.insert(6, "n_input_genes", len(genes))
            frames.append(work)

    if frames:
        result = pd.concat(frames, ignore_index=True).sort_values(["library_key", "direction", "Adjusted P-value"]).reset_index(drop=True)
    else:
        result = pd.DataFrame(columns=["cell_type_query", "cell_type_sheet", "level", "direction", "library_key", "library_label", "n_input_genes", "Term", "Adjusted P-value"])

    if save_csv:
        CONFIG.selective_dir.mkdir(parents=True, exist_ok=True)
        out_path = CONFIG.selective_dir / f"{sanitize_name(match['sheet_name'])}__{match['level']}__{direction}__pathway_nb12.csv"
        result.to_csv(out_path, index=False)

        row = pd.DataFrame([
            {
                "timestamp_utc": datetime.now(timezone.utc).isoformat(),
                "cell_type_query": match["query"],
                "cell_type_sheet": match["sheet_name"],
                "level": match["level"],
                "direction": direction,
                "libraries": "|".join(library_keys),
                "rows": int(len(result)),
                "output_csv": str(out_path),
            }
        ])
        CONFIG.selective_manifest.parent.mkdir(parents=True, exist_ok=True)
        if CONFIG.selective_manifest.exists():
            old = pd.read_csv(CONFIG.selective_manifest)
            pd.concat([old, row], ignore_index=True).to_csv(CONFIG.selective_manifest, index=False)
        else:
            row.to_csv(CONFIG.selective_manifest, index=False)

    return result


def normalize_gsea_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out = out.rename(columns={
        "Term": "term",
        "NES": "nes",
        "FDR q-val": "fdr_qval",
        "NOM p-val": "nominal_pval",
    })
    required = ["term", "nes", "fdr_qval"]
    missing = [c for c in required if c not in out.columns]
    if missing:
        raise KeyError(f"Missing required columns: {missing}")
    out["nes"] = pd.to_numeric(out["nes"], errors="coerce")
    out["fdr_qval"] = pd.to_numeric(out["fdr_qval"], errors="coerce")
    out["neg_log10_fdr"] = -np.log10(out["fdr_qval"].clip(lower=np.finfo(float).tiny))
    return out.dropna(subset=["term", "nes", "fdr_qval"]).copy()


def level_csv_candidates(level: str) -> list[Path]:
    return [
        CONFIG.enrichment_root / f"level_{level}" / f"gsea_level_{level}.csv",
        CONFIG.analysis_root / "Results" / "GSEA" / f"level_{level}" / f"gsea_level_{level}.csv",
    ]


def load_gsea_level(level: str) -> pd.DataFrame:
    if level not in LEVELS:
        raise ValueError(f"Unsupported level={level!r}. Expected one of {LEVELS}")
    for path in level_csv_candidates(level):
        if path.exists():
            return normalize_gsea_columns(pd.read_csv(path))
    attempted = "\n".join(str(p) for p in level_csv_candidates(level))
    raise FileNotFoundError(f"Missing level GSEA CSV for level={level}. Tried:\n{attempted}")


def _save_and_show(fig: Figure, output_path: Path | None, show: bool) -> None:
    if output_path is not None:
        output_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(str(output_path), dpi=200, bbox_inches="tight")
        print(f"Saved plot: {output_path}")
    if show:
        plt.show()
    else:
        plt.close(fig)


def plot_top_nes_bar(level: str, top_n: int = 20, fdr_max: float = 0.25, output_path: Path | None = None, show: bool = True) -> pd.DataFrame:
    df = load_gsea_level(level)
    df = df[df["fdr_qval"] <= fdr_max].copy()
    if df.empty:
        raise ValueError(f"No pathways pass fdr_max={fdr_max} for level={level}")

    pos = df[df["nes"] > 0].nlargest(top_n, "nes")
    neg = df[df["nes"] < 0].nsmallest(top_n, "nes")
    plot_df = pd.concat([neg, pos], ignore_index=True)
    plot_df["direction"] = np.where(plot_df["nes"] > 0, "positive NES", "negative NES")
    plot_df = plot_df.sort_values("nes")

    fig, ax = plt.subplots(figsize=(11, max(6, 0.35 * len(plot_df))))
    sns.barplot(data=plot_df, y="term", x="nes", hue="direction", dodge=False, palette={"positive NES": "#c44e52", "negative NES": "#4c72b0"}, ax=ax)
    ax.axvline(0, color="#333333", lw=1)
    ax.set_title(f"Top GSEA Pathways by NES | level={level} | FDR<={fdr_max}")
    ax.set_xlabel("NES")
    ax.set_ylabel("pathway")
    ax.legend(title="direction", loc="lower right")
    fig.tight_layout()

    if output_path is None:
        output_path = CONFIG.plot_dir / f"level_{level}__top_nes_bar_nb12.png"
    _save_and_show(fig, output_path=output_path, show=show)
    return plot_df


def plot_nes_vs_fdr(level: str, fdr_max: float = 0.25, output_path: Path | None = None, show: bool = True) -> pd.DataFrame:
    plot_df = load_gsea_level(level).copy()
    plot_df["pass_fdr"] = plot_df["fdr_qval"] <= fdr_max

    fig, ax = plt.subplots(figsize=(9, 6))
    sns.scatterplot(data=plot_df, x="nes", y="neg_log10_fdr", hue="pass_fdr", palette={True: "#c44e52", False: "#9aa0a6"}, alpha=0.9, s=24, linewidth=0, ax=ax)
    ax.axhline(-np.log10(fdr_max), ls="--", lw=1, color="#333333")
    ax.axvline(0, ls="--", lw=1, color="#333333")
    ax.set_title(f"GSEA NES vs -log10(FDR) | level={level}")
    ax.set_xlabel("NES")
    ax.set_ylabel("-log10(FDR q-value)")
    ax.legend(title=f"FDR <= {fdr_max}", loc="upper right")
    fig.tight_layout()

    if output_path is None:
        output_path = CONFIG.plot_dir / f"level_{level}__nes_vs_fdr_nb12.png"
    _save_and_show(fig, output_path=output_path, show=show)
    return plot_df


def build_rank_series_from_dge(dge_df: pd.DataFrame) -> pd.Series:
    gene_col = infer_first_existing(dge_df.columns, ["gene", "Gene", "names", "gene_symbol"], "gene")
    pval_col = infer_first_existing(dge_df.columns, ["p_adj", "p_adj_scanpy", "pvals_adj", "adj.P.Val", "FDR", "p_val_adj", "p_val"], "p-value")
    effect_col = infer_first_existing(dge_df.columns, ["logFC", "avg_log2FC", "log2FoldChange", "logfoldchanges", "avg_logFC"], "effect")

    rank_df = dge_df[[gene_col, pval_col, effect_col]].copy()
    rank_df[gene_col] = rank_df[gene_col].astype(str).str.strip()
    rank_df[pval_col] = pd.to_numeric(rank_df[pval_col], errors="coerce").fillna(1.0).clip(lower=np.finfo(float).tiny)
    rank_df[effect_col] = pd.to_numeric(rank_df[effect_col], errors="coerce")
    rank_df = rank_df.dropna(subset=[gene_col, effect_col]).copy()
    rank_df["rank_score"] = -np.log10(rank_df[pval_col]) * np.sign(rank_df[effect_col])
    rank_df = rank_df.sort_values(["rank_score", gene_col], ascending=[False, True]).drop_duplicates(subset=[gene_col])
    return pd.Series(rank_df["rank_score"].to_numpy(), index=rank_df[gene_col].to_numpy())


def run_custom_gene_list_gsea(level: str, cell_type_name: str, genes: list[str], gene_set_name: str = "custom_gene_list", run_label: str | None = None, permutations: int = 1000) -> pd.DataFrame:
    try:
        import gseapy as gp
    except ImportError as exc:
        raise ImportError("gseapy is required for custom prerank GSEA.") from exc

    cleaned = []
    seen: set[str] = set()
    for gene in genes:
        gene_name = str(gene).strip()
        if gene_name and gene_name not in seen:
            seen.add(gene_name)
            cleaned.append(gene_name)
    if not cleaned:
        raise ValueError("genes must contain at least one non-empty unique symbol")

    match = locate_cell_type(cell_type=cell_type_name, level=level)
    dge_df = read_celltype_dge(match)
    rank_series = build_rank_series_from_dge(dge_df)

    safe_label = sanitize_name(run_label or f"{level}_{cell_type_name}_{gene_set_name}")
    out_dir = CONFIG.custom_dir / safe_label
    out_dir.mkdir(parents=True, exist_ok=True)

    result = gp.prerank(
        rnk=rank_series,
        gene_sets={gene_set_name: cleaned},  # type: ignore[arg-type]
        min_size=max(1, len(cleaned)),
        max_size=max(1, len(cleaned)),
        permutation_num=permutations,
        outdir=str(out_dir),
        seed=42,
        verbose=True,
    )

    result_df = getattr(result, "res2d", None)
    if result_df is None:
        report_csv = out_dir / "gseapy.gene_set.prerank.report.csv"
        if not report_csv.exists():
            raise RuntimeError(f"Custom GSEA report missing for run_label={safe_label}")
        result_df = pd.read_csv(report_csv)

    out_csv = out_dir / f"gsea_custom_{safe_label}.csv"
    result_df.to_csv(out_csv, index=False)
    print(f"Saved custom GSEA table: {out_csv}")
    return normalize_gsea_columns(result_df)


print("Notebook 12 merged setup loaded.")
print(f"DGE directory: {CONFIG.dge_dir}")
print(f"Enrichment root: {CONFIG.enrichment_root}")
print(f"Selective outputs: {CONFIG.selective_dir}")
print(f"Plot outputs: {CONFIG.plot_dir}")

## Selective Enrichment Example

Use this section to run selective pathway enrichment for one cell type from saved DGE workbooks.

In [ ]:
TARGET_LEVEL = "class"
TARGET_CELL = "01 IT-ET Glut"
RUN_SELECTIVE_ENRICHMENT = False

print(f"Preview of available cell types at level={TARGET_LEVEL}:")
display(list_available_cell_types(level=TARGET_LEVEL).head(20))

if RUN_SELECTIVE_ENRICHMENT:
    selective_df = run_celltype_pathway_enrichment(
        cell_type=TARGET_CELL,
        level=TARGET_LEVEL,
        libraries=["go", "kegg", "reactome", "wiki", "jensen"],
        direction="both",
        p_adj_threshold=0.05,
        min_abs_logfc=0.25,
        max_genes=300,
        fdr_threshold=0.05,
        top_n_per_library=25,
        organism="mouse",
        save_csv=True,
    )
    print("Selective enrichment results:")
    display(selective_df.head(30))
else:
    print("RUN_SELECTIVE_ENRICHMENT is False. Set to True to execute selective enrichment.")

## Visualization and Optional Custom Gene-List GSEA

Use level-wise plotting (from saved CSVs) and optionally run custom gene-list prerank GSEA for one level/cell type.

In [ ]:
PLOT_LEVEL = "class"
RUN_PLOTTING = False
RUN_CUSTOM_GENE_LIST_GSEA = False

if RUN_PLOTTING:
    top_plot_df = plot_top_nes_bar(level=PLOT_LEVEL, top_n=15, fdr_max=0.25, show=True)
    scatter_df = plot_nes_vs_fdr(level=PLOT_LEVEL, fdr_max=0.25, show=True)
    display(top_plot_df.head(20))
    display(scatter_df.head(20))
else:
    print("RUN_PLOTTING is False. Set to True to generate level-wise pathway plots.")

CUSTOM_LEVEL = "class"
CUSTOM_CELL_TYPE = "01 IT-ET Glut"
CUSTOM_GENE_SET_NAME = "example_interferon_genes"
CUSTOM_GENES = ["Stat1", "Stat2", "Irf7", "Isg15", "Ifit1"]

if RUN_CUSTOM_GENE_LIST_GSEA:
    custom_results = run_custom_gene_list_gsea(
        level=CUSTOM_LEVEL,
        cell_type_name=CUSTOM_CELL_TYPE,
        genes=CUSTOM_GENES,
        gene_set_name=CUSTOM_GENE_SET_NAME,
        permutations=1000,
    )
    cols = [column for column in ["term", "es", "nes", "fdr_qval"] if column in custom_results.columns]
    display(custom_results[cols].head(20) if cols else custom_results.head(20))
else:
    print("RUN_CUSTOM_GENE_LIST_GSEA is False. Set to True to run custom gene-list prerank GSEA.")